# Extended Kalman Filter: Robot Arm and Sensor Fusion

## Learning Objectives

- Apply the EKF to a **2-DOF planar robot arm** (4-state: joint angles + velocities)
- Understand **forward kinematics**: how joint angles map to end-effector position
- Design **measurement models** for different sensor types (encoders vs vision)
- Implement **multi-rate sensor fusion** (fast encoders + slow vision)

## What's New vs. Previous Notebooks?

In Notebooks 01-03, we worked with systems where all sensors measured the same type of quantity (joint angles/positions). Now we tackle a practical robotics challenge: **different sensors measure different things at different rates**.

| Sensor Type | What It Measures | Update Rate | Noise Level |
|-------------|-----------------|-------------|-------------|
| Joint Encoders | Joint angles directly | 100 Hz (fast) | Low (precise) |
| Vision System | End-effector position (Cartesian) | 10 Hz (slow) | Higher (noisy) |

The **multi-rate fusion pattern** is simple but powerful:
1. Predict at the fastest sensor rate (encoder rate)
2. Update with encoder measurements (available every step)
3. Update with vision measurements only when available (every 10th step)

This notebook shows that the same EKF predict/update functions handle this naturally -- no special algorithm required.

## Section 2: Setup

We import the same libraries as previous notebooks and load the 2-DOF planar robot arm model.

In [1]:
import numpy as np
import mujoco
import matplotlib.pyplot as plt
import mediapy as media
from scipy.optimize import approx_fprime
import ipywidgets
from ipywidgets import interact_manual, FloatLogSlider, IntSlider

# Load the planar arm model
model = mujoco.MjModel.from_xml_path('../models/planar_arm.xml')
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

print(f'Model loaded: {model.nq} position DOFs, {model.nv} velocity DOFs')
print(f'Timestep: {model.opt.timestep} s')
print(f'Gravity: {model.opt.gravity}')
print()
print('State convention: x = [theta1, omega1, theta2, omega2]^T  (4,1)')
print('  theta1 = shoulder angle (from positive X axis, rad)')
print('  omega1 = shoulder angular velocity (rad/s)')
print('  theta2 = elbow angle (relative to link 1, rad)')
print('  omega2 = elbow angular velocity (rad/s)')
print()
print('Link lengths: L1 = 0.4 m, L2 = 0.3 m')
print('Total reach: 0.7 m (arm fully extended)')

Model loaded: 2 position DOFs, 2 velocity DOFs
Timestep: 0.002 s
Gravity: [ 0.    0.   -9.81]

State convention: x = [theta1, omega1, theta2, omega2]^T  (4,1)
  theta1 = shoulder angle (from positive X axis, rad)
  omega1 = shoulder angular velocity (rad/s)
  theta2 = elbow angle (relative to link 1, rad)
  omega2 = elbow angular velocity (rad/s)

Link lengths: L1 = 0.4 m, L2 = 0.3 m
Total reach: 0.7 m (arm fully extended)


## Section 3: Robot Arm Simulation

A 2-DOF planar arm moves in the XY plane (horizontal). The shoulder joint rotates link 1 around a vertical axis (Z), and the elbow joint rotates link 2 relative to link 1. Since the arm moves horizontally, gravity has no effect on the joint torques (unlike a pendulum that swings vertically).

We simulate 5 seconds of free dynamics with damping. The arm starts with both joints displaced from zero, and it gradually returns to rest as damping dissipates the energy.

In [2]:
# Simulation parameters
duration = 5.0       # seconds
framerate = 30       # Hz for video capture
dt = model.opt.timestep  # 0.002 s

# Reset and set initial conditions
mujoco.mj_resetData(model, data)
data.qpos[0] = 0.3    # Shoulder angle: 0.3 rad (~17 degrees from X axis)
data.qpos[1] = -0.5   # Elbow angle: -0.5 rad (~29 degrees, bent inward)
data.qvel[0] = 0.5    # Initial shoulder velocity (to see motion)
data.qvel[1] = -0.3   # Initial elbow velocity
mujoco.mj_forward(model, data)  # Compute initial derived quantities

# State vector: x = [theta1, omega1, theta2, omega2]^T
# MuJoCo mapping: x[0] = qpos[0], x[1] = qvel[0], x[2] = qpos[1], x[3] = qvel[1]
true_states = []   # List of (4,1) column vectors
times = []         # Time values
frames = []        # Video frames

# Run simulation
while data.time < duration:
    # Store ground truth BEFORE stepping (use .copy() to avoid MuJoCo buffer mutation)
    state = np.array([[data.qpos[0].copy()],   # theta1 (shoulder)
                      [data.qvel[0].copy()],   # omega1 (shoulder velocity)
                      [data.qpos[1].copy()],   # theta2 (elbow)
                      [data.qvel[1].copy()]])  # omega2 (elbow velocity)
    true_states.append(state)
    times.append(data.time)

    # Capture video frame at desired framerate
    if len(frames) < data.time * framerate:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

    # Step physics
    mujoco.mj_step(model, data)

times = np.array(times)
print(f'Simulation complete: {len(true_states)} timesteps, {len(frames)} video frames')
print(f'Time range: {times[0]:.3f} to {times[-1]:.3f} s')
print(f'State shape: {true_states[0].shape} (4x1 column vector)')
print(f'Final joint angles: theta1={true_states[-1][0,0]:.4f} rad, theta2={true_states[-1][2,0]:.4f} rad')

Simulation complete: 2501 timesteps, 150 video frames
Time range: 0.000 to 5.000 s
State shape: (4, 1) (4x1 column vector)
Final joint angles: theta1=0.3155 rad, theta2=-0.3352 rad


In [4]:
# Display robot arm simulation video (top-down view)
media.show_video(frames, fps=framerate)

## Section 4: Forward Kinematics

**Forward kinematics** computes where the end-effector is given the joint angles. This is how we design the vision sensor measurement model -- the camera sees the end-effector position in Cartesian coordinates (x, y), not the joint angles directly.

For a 2-link planar arm:

$$x_{ee} = L_1 \cos(\theta_1) + L_2 \cos(\theta_1 + \theta_2)$$
$$y_{ee} = L_1 \sin(\theta_1) + L_2 \sin(\theta_1 + \theta_2)$$

where:
- $L_1 = 0.4$ m (upper arm length)
- $L_2 = 0.3$ m (forearm length)
- $\theta_1$ = shoulder angle from positive X axis
- $\theta_2$ = elbow angle relative to link 1

In [ ]:
# Link lengths (must match planar_arm.xml)
L1 = 0.4  # Upper arm (shoulder to elbow)
L2 = 0.3  # Forearm (elbow to end-effector)


def forward_kinematics(theta1, theta2, l1=L1, l2=L2):
    """
    Compute end-effector position from joint angles.
    
    theta1: shoulder angle (rad, measured from positive X axis)
    theta2: elbow angle (rad, relative to link 1)
    Returns: (x, y) end-effector position
    """
    x = l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2)
    y = l1 * np.sin(theta1) + l2 * np.sin(theta1 + theta2)
    return x, y


# Test: compute end-effector position for initial joint angles
theta1_init = 0.3
theta2_init = -0.5
x_ee, y_ee = forward_kinematics(theta1_init, theta2_init)
print(f'Initial joint angles: theta1={theta1_init:.3f} rad, theta2={theta2_init:.3f} rad')
print(f'Computed end-effector position: x={x_ee:.4f} m, y={y_ee:.4f} m')

In [ ]:
# Verify forward kinematics against MuJoCo's site position
# Reset to known state and compare
mujoco.mj_resetData(model, data)
data.qpos[0] = theta1_init
data.qpos[1] = theta2_init
mujoco.mj_forward(model, data)

# MuJoCo site position (end_effector site)
site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, 'end_effector')
mujoco_x = data.site_xpos[site_id][0]
mujoco_y = data.site_xpos[site_id][1]

print(f'Computed (forward_kinematics): x={x_ee:.6f}, y={y_ee:.6f}')
print(f'MuJoCo (site_xpos):            x={mujoco_x:.6f}, y={mujoco_y:.6f}')
print()
error = np.sqrt((x_ee - mujoco_x)**2 + (y_ee - mujoco_y)**2)
print(f'Position error: {error:.6e} m')
assert error < 1e-4, f'Forward kinematics error too large: {error}'
print('Forward kinematics VERIFIED (error < 1e-4 m)')

In [ ]:
# Plot arm configuration at initial state
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: single arm configuration
ax = axes[0]
# Joint positions
x0, y0 = 0, 0  # Base (shoulder)
x1, y1 = L1 * np.cos(theta1_init), L1 * np.sin(theta1_init)  # Elbow
x2, y2 = x_ee, y_ee  # End-effector

# Draw links
ax.plot([x0, x1], [y0, y1], 'r-', lw=4, label='Link 1 (upper arm)')
ax.plot([x1, x2], [y1, y2], 'b-', lw=3, label='Link 2 (forearm)')

# Draw joints and end-effector
ax.scatter([x0], [y0], s=200, c='gray', zorder=5, label='Base (shoulder)')
ax.scatter([x1], [y1], s=150, c='orange', zorder=5, label='Elbow')
ax.scatter([x2], [y2], s=100, c='green', zorder=5, label='End-effector')

ax.set_xlim(-0.8, 0.8)
ax.set_ylim(-0.8, 0.8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title(f'Arm Configuration (theta1={theta1_init:.2f}, theta2={theta2_init:.2f})')
ax.legend(loc='upper left')

# Right plot: workspace (reachable end-effector positions)
ax = axes[1]
# Sweep joint angles and plot reachable positions
theta1_range = np.linspace(-np.pi, np.pi, 100)
theta2_range = np.linspace(-np.pi, np.pi, 100)
workspace_x = []
workspace_y = []
for t1 in theta1_range:
    for t2 in theta2_range:
        x, y = forward_kinematics(t1, t2)
        workspace_x.append(x)
        workspace_y.append(y)

ax.scatter(workspace_x, workspace_y, s=1, c='lightblue', alpha=0.3, label='Reachable workspace')

# Mark current end-effector position
ax.scatter([x_ee], [y_ee], s=100, c='red', zorder=5, label='Current position')

# Draw workspace boundaries (circles)
theta_circle = np.linspace(0, 2*np.pi, 100)
# Outer boundary: arm fully extended
ax.plot((L1+L2)*np.cos(theta_circle), (L1+L2)*np.sin(theta_circle), 'k--', lw=1, label=f'Max reach ({L1+L2} m)')
# Inner boundary: arm fully folded
ax.plot(abs(L1-L2)*np.cos(theta_circle), abs(L1-L2)*np.sin(theta_circle), 'k:', lw=1, label=f'Min reach ({abs(L1-L2)} m)')

ax.set_xlim(-0.9, 0.9)
ax.set_ylim(-0.9, 0.9)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Workspace: Reachable End-Effector Positions')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

print('Left: Current arm configuration showing links and joints.')
print('Right: The annular workspace -- the arm can reach any point between the inner and outer circles.')

## Section 5: Robot Arm Dynamics Model

The arm dynamics are simpler than the double pendulum because the base is fixed and the arm moves in a horizontal plane (no gravity torques). We use a simplified model treating each link as a point mass at the end.

Since the arm moves in the XY plane (horizontal), **gravity has no effect** on a truly horizontal arm -- the gravitational force acts in the Z direction, which is perpendicular to the plane of motion. The only dynamics are from damping at the joints.

### Physical Parameters

| Parameter | Symbol | Value | Notes |
|-----------|--------|-------|-------|
| Link 1 mass | $m_1$ | 1.0 kg | Upper arm |
| Link 2 mass | $m_2$ | 0.5 kg | Forearm |
| Link 1 length | $L_1$ | 0.4 m | Shoulder to elbow |
| Link 2 length | $L_2$ | 0.3 m | Elbow to end-effector |
| Joint damping | $b$ | 0.1 N m s/rad | Same for both joints |

### Simplified Dynamics

For a horizontal arm with no gravity torques, the dominant effect is damping:

$$\ddot{\theta}_1 = -b \cdot \dot{\theta}_1$$
$$\ddot{\theta}_2 = -b \cdot \dot{\theta}_2$$

This is a simplification -- the full dynamics would include inertial coupling between joints. But for teaching sensor fusion, this simplified model (combined with Q to absorb model error) is sufficient.

In [ ]:
# Physical parameters (must match planar_arm.xml)
m1 = 1.0   # Link 1 mass (kg)
m2 = 0.5   # Link 2 mass (kg)
b = 0.1    # Joint damping (N*m*s/rad)


def arm_f(x, dt):
    """
    State transition function for 2-DOF planar arm.
    
    x: (4,1) state vector [theta1, omega1, theta2, omega2]^T
    dt: time step (seconds)
    Returns: (4,1) predicted state
    
    Simplified dynamics: damped oscillator (no gravity torque for horizontal arm)
    """
    theta1    = x[0, 0]
    omega1    = x[1, 0]
    theta2    = x[2, 0]
    omega2    = x[3, 0]
    
    # Simple damped dynamics (no gravity for horizontal arm)
    # Full dynamics would include inertial coupling, but we simplify for teaching
    alpha1 = -b * omega1  # Angular acceleration of shoulder
    alpha2 = -b * omega2  # Angular acceleration of elbow
    
    # Euler integration
    theta1_new = theta1 + dt * omega1
    omega1_new = omega1 + dt * alpha1
    theta2_new = theta2 + dt * omega2
    omega2_new = omega2 + dt * alpha2
    
    return np.array([[theta1_new], [omega1_new], [theta2_new], [omega2_new]])


# Quick sanity check
x_test = np.array([[0.3], [0.5], [-0.5], [-0.3]])
x_next = arm_f(x_test, dt)
print(f'Test state:     {x_test.T}')
print(f'Predicted next: {x_next.T}')
print()
print('Simplified dynamics: damping gradually reduces velocities to zero.')
print('The Q matrix will account for model mismatch with MuJoCo.')

In [ ]:
# Compare dynamics model against MuJoCo ground truth
# Run our simplified model forward from the same initial conditions
x_model = np.array([[0.3], [0.5], [-0.5], [-0.3]])  # Same initial conditions as simulation
model_states = [x_model.copy()]

for k in range(len(true_states) - 1):
    x_model = arm_f(x_model, dt)
    model_states.append(x_model.copy())

# Plot comparison: 4 subplots showing all states
labels = ['Shoulder Angle (rad)', 'Shoulder Velocity (rad/s)',
          'Elbow Angle (rad)', 'Elbow Velocity (rad/s)']

fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

for i in range(4):
    true_vals = [s[i, 0] for s in true_states]
    model_vals = [s[i, 0] for s in model_states]

    axes[i].plot(times, true_vals, 'b-', lw=2, label='MuJoCo (ground truth)')
    axes[i].plot(times, model_vals, 'r--', lw=2, label='Simplified model')
    axes[i].set_ylabel(labels[i])
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Dynamics Model vs MuJoCo Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('The simplified model captures the general behavior (damped motion) but drifts from MuJoCo.')
print('This is expected: our model ignores inertial coupling between joints.')
print('The Q matrix in our EKF will account for this model mismatch.')

## Section 6: Numerical Jacobian

Same pattern as previous notebooks: we compute the 4x4 Jacobian numerically using finite differences.

In [ ]:
def arm_F_numerical(x, dt, eps=1e-7):
    """
    Compute the 4x4 Jacobian numerically using finite differences.
    
    x: (4,1) state vector
    dt: time step
    eps: perturbation size for finite differences
    Returns: (4,4) Jacobian matrix F = df/dx
    """
    n = x.shape[0]
    F = np.zeros((n, n))
    x_flat = x.flatten()
    for i in range(n):
        def f_i(x_flat_in, idx=i):
            x_col = x_flat_in.reshape(-1, 1)
            return arm_f(x_col, dt)[idx, 0]
        F[i, :] = approx_fprime(x_flat, f_i, eps)
    return F


# Verify Jacobian at multiple test states
test_states_jac = [
    ('All zeros',                    np.array([[0.0], [0.0], [0.0],  [0.0]])),
    ('Small angles',                 np.array([[0.3], [0.0], [-0.5], [0.0]])),
    ('With velocities',              np.array([[0.3], [0.5], [-0.5], [-0.3]])),
    ('Large angles',                 np.array([[1.5], [1.0], [1.0],  [-0.5]])),
]

print('Jacobian Verification (numerical eps=1e-7 vs eps=1e-5)')
print('=' * 70)
all_passed = True
for name, x_test in test_states_jac:
    F1 = arm_F_numerical(x_test, dt, eps=1e-7)
    F2 = arm_F_numerical(x_test, dt, eps=1e-5)
    error = np.max(np.abs(F1 - F2))
    status = 'PASS' if error < 1e-4 else 'FAIL'
    if error >= 1e-4:
        all_passed = False
    print(f'\n{name}:')
    print(f'  Max difference: {error:.2e} [{status}]')

print('\n' + '=' * 70)
assert all_passed, 'Jacobian verification FAILED at one or more test points!'
print('All Jacobian tests PASSED (difference < 1e-4)')

In [ ]:
# Examine the Jacobian structure at equilibrium
x_eq = np.array([[0.0], [0.0], [0.0], [0.0]])
F_eq = arm_F_numerical(x_eq, dt)

print('Jacobian at equilibrium (all states zero):')
print()
np.set_printoptions(precision=6, suppress=True)
print(F_eq)
print()
print('Structure explanation:')
print('  Row 0 (theta1):  [1, dt, 0, 0]    -- angle updates by velocity')
print('  Row 1 (omega1):  [0, 1-b*dt, 0, 0] -- velocity decays by damping')
print('  Row 2 (theta2):  [0, 0, 1, dt]    -- angle updates by velocity')
print('  Row 3 (omega2):  [0, 0, 0, 1-b*dt] -- velocity decays by damping')
print()
print('Key observation: The simplified dynamics are decoupled (block diagonal).')
print('MuJoCo has coupling from inertia -- Q will absorb this mismatch.')

## Section 7: Measurement Models - Encoders vs Vision

Different sensors measure different things. **Joint encoders** measure angles directly (fast, precise). **Vision systems** measure end-effector position in Cartesian coordinates (slower, noisier, but measures what we often care about).

This is the key insight for sensor fusion: each sensor has different strengths.

### Encoder Measurement Model (Joint Space)

Encoders measure joint angles directly:
$$\mathbf{z}_{encoder} = \begin{bmatrix} \theta_1 \\ \theta_2 \end{bmatrix}$$

The measurement Jacobian is constant (linear measurement):
$$\mathbf{H}_{encoder} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 0 & 1 & 0 \end{bmatrix}$$

### Vision Measurement Model (Cartesian Space)

Vision measures end-effector position via forward kinematics:
$$\mathbf{z}_{vision} = \begin{bmatrix} x_{ee} \\ y_{ee} \end{bmatrix} = \begin{bmatrix} L_1 \cos(\theta_1) + L_2 \cos(\theta_1 + \theta_2) \\ L_1 \sin(\theta_1) + L_2 \sin(\theta_1 + \theta_2) \end{bmatrix}$$

The measurement Jacobian depends on configuration (nonlinear measurement):
$$\mathbf{H}_{vision} = \begin{bmatrix} \frac{\partial x}{\partial \theta_1} & 0 & \frac{\partial x}{\partial \theta_2} & 0 \\ \frac{\partial y}{\partial \theta_1} & 0 & \frac{\partial y}{\partial \theta_2} & 0 \end{bmatrix}$$

where:
- $\frac{\partial x}{\partial \theta_1} = -L_1 \sin(\theta_1) - L_2 \sin(\theta_1 + \theta_2)$
- $\frac{\partial x}{\partial \theta_2} = -L_2 \sin(\theta_1 + \theta_2)$
- $\frac{\partial y}{\partial \theta_1} = L_1 \cos(\theta_1) + L_2 \cos(\theta_1 + \theta_2)$
- $\frac{\partial y}{\partial \theta_2} = L_2 \cos(\theta_1 + \theta_2)$

In [ ]:
# ============== ENCODER MEASUREMENT MODEL ==============

def h_encoder(x):
    """
    Encoder measurement: observe joint angles directly.
    x: (4,1) state [theta1, omega1, theta2, omega2]^T
    Returns: (2,1) measurement [theta1, theta2]^T
    """
    return np.array([[x[0, 0]], [x[2, 0]]])


def H_encoder(x):
    """
    Encoder Jacobian: constant (linear measurement).
    Returns: (2,4) Jacobian
    """
    return np.array([[1, 0, 0, 0],
                     [0, 0, 1, 0]])


# Encoder noise: very precise (0.01 rad ~ 0.5 degrees)
encoder_noise_std = 0.01  # rad
R_encoder = np.diag([encoder_noise_std**2, encoder_noise_std**2])


# ============== VISION MEASUREMENT MODEL ==============

def h_vision(x, l1=L1, l2=L2):
    """
    Vision measurement: end-effector position via forward kinematics.
    x: (4,1) state [theta1, omega1, theta2, omega2]^T
    Returns: (2,1) measurement [x_ee, y_ee]^T
    """
    theta1 = x[0, 0]
    theta2 = x[2, 0]
    x_ee = l1 * np.cos(theta1) + l2 * np.cos(theta1 + theta2)
    y_ee = l1 * np.sin(theta1) + l2 * np.sin(theta1 + theta2)
    return np.array([[x_ee], [y_ee]])


def H_vision(x, l1=L1, l2=L2):
    """
    Vision Jacobian: depends on configuration (nonlinear).
    x: (4,1) state
    Returns: (2,4) Jacobian
    """
    theta1 = x[0, 0]
    theta2 = x[2, 0]
    
    s1 = np.sin(theta1)
    c1 = np.cos(theta1)
    s12 = np.sin(theta1 + theta2)
    c12 = np.cos(theta1 + theta2)
    
    # Partial derivatives
    # dx/dtheta1 = -L1*sin(theta1) - L2*sin(theta1+theta2)
    # dx/dtheta2 = -L2*sin(theta1+theta2)
    # dy/dtheta1 = L1*cos(theta1) + L2*cos(theta1+theta2)
    # dy/dtheta2 = L2*cos(theta1+theta2)
    # Velocities don't affect position: dx/domega = dy/domega = 0
    
    H = np.array([
        [-l1*s1 - l2*s12, 0, -l2*s12, 0],  # dx/d[theta1, omega1, theta2, omega2]
        [ l1*c1 + l2*c12, 0,  l2*c12, 0]   # dy/d[theta1, omega1, theta2, omega2]
    ])
    return H


# Vision noise: noisier than encoders (1mm ~ 0.001m position uncertainty)
vision_noise_std = 0.001  # m
R_vision = np.diag([vision_noise_std**2, vision_noise_std**2])


# ============== PRINT SUMMARY ==============
print('Measurement Model Summary')
print('=' * 60)
print()
print('ENCODER (joint space):')
print(f'  h_encoder(x) -> (2,1): [theta1, theta2]')
print(f'  H_encoder    -> (2,4): constant matrix')
print(f'  R_encoder    -> (2,2): diag([{encoder_noise_std**2:.0e}, {encoder_noise_std**2:.0e}])')
print(f'  Noise level: {encoder_noise_std} rad (~{np.degrees(encoder_noise_std):.1f} degrees)')
print()
print('VISION (Cartesian space):')
print(f'  h_vision(x)  -> (2,1): [x_ee, y_ee] via forward kinematics')
print(f'  H_vision(x)  -> (2,4): configuration-dependent')
print(f'  R_vision     -> (2,2): diag([{vision_noise_std**2:.0e}, {vision_noise_std**2:.0e}])')
print(f'  Noise level: {vision_noise_std*1000} mm')
print()
print('Key difference: Encoder H is constant, Vision H varies with arm configuration.')

In [ ]:
# Demonstrate that vision Jacobian varies with configuration
test_configs = [
    ('Arm extended (theta1=0, theta2=0)',    np.array([[0.0], [0.0], [0.0],  [0.0]])),
    ('Arm at 45 deg (theta1=pi/4, theta2=0)', np.array([[np.pi/4], [0.0], [0.0], [0.0]])),
    ('Arm bent (theta1=0, theta2=pi/2)',     np.array([[0.0], [0.0], [np.pi/2], [0.0]])),
]

print('Vision Jacobian H_vision at different configurations:')
print('=' * 60)

for name, x_test in test_configs:
    H_v = H_vision(x_test)
    print(f'\n{name}:')
    print(f'  H_vision = ')
    print(f'    [{H_v[0,0]:7.4f}, {H_v[0,1]:7.4f}, {H_v[0,2]:7.4f}, {H_v[0,3]:7.4f}]  (dx/d[states])')
    print(f'    [{H_v[1,0]:7.4f}, {H_v[1,1]:7.4f}, {H_v[1,2]:7.4f}, {H_v[1,3]:7.4f}]  (dy/d[states])')

print('\n' + '=' * 60)
print('Notice: The Jacobian changes with configuration.')
print('This is why we need the Extended Kalman Filter (EKF) -- the measurement is nonlinear.')

## Section 8: EKF Functions + Single-Sensor Baselines

The EKF predict and update functions are identical to all previous notebooks -- they are dimension-agnostic. The only change is which measurement model we use.

Before implementing multi-rate fusion, we establish **single-sensor baselines**: how well does the EKF perform with only encoders? Only vision? This gives us a reference point to see if fusion actually helps.

In [ ]:
# ============== EKF FUNCTIONS (identical to previous notebooks) ==============

def ekf_predict(x, P, f, F, Q, dt):
    """
    EKF Prediction Step - works for any state dimension.
    """
    x_pred = f(x, dt)
    F_k = F(x, dt)
    P_pred = F_k @ P @ F_k.T + Q
    return x_pred, P_pred


def ekf_update(x, P, z, h, H, R):
    """
    EKF Update Step with Joseph form - works for any dimensions.
    """
    H_k = H(x)
    y = z - h(x)                          # Innovation
    S = H_k @ P @ H_k.T + R               # Innovation covariance
    K = P @ H_k.T @ np.linalg.inv(S)       # Kalman gain
    x_upd = x + K @ y                      # State update
    I_KH = np.eye(x.shape[0]) - K @ H_k
    P_upd = I_KH @ P @ I_KH.T + K @ R @ K.T  # Joseph form
    return x_upd, P_upd


print('EKF functions defined: ekf_predict, ekf_update')
print('  - Identical to all previous notebooks (dimension-agnostic)')
print('  - Uses Joseph form for numerically stable covariance update')

In [ ]:
# ============== EKF INITIALIZATION ==============

# Process noise: higher on velocities (model mismatch primarily in acceleration)
Q = np.diag([1e-4, 1e-2, 1e-4, 1e-2])

# Initial state estimate: zero (no prior knowledge)
x0 = np.zeros((4, 1))

# Initial covariance: more uncertainty on velocities
P0 = np.diag([0.1, 1.0, 0.1, 1.0])

# Wrap Jacobian for ekf_predict signature
F_func = lambda x, dt: arm_F_numerical(x, dt)

print('EKF Initialization:')
print(f'  x0 = {x0.T}')
print(f'  P0 = diag{list(np.diag(P0))}')
print(f'  Q  = diag{list(np.diag(Q))}')

In [ ]:
# ============== GENERATE NOISY MEASUREMENTS ==============

np.random.seed(42)  # For reproducibility

# Encoder measurements (for all timesteps)
encoder_measurements = []
for state in true_states:
    z_true = h_encoder(state)
    noise = np.array([[np.random.randn() * encoder_noise_std],
                      [np.random.randn() * encoder_noise_std]])
    encoder_measurements.append(z_true + noise)

# Vision measurements (for all timesteps)
vision_measurements = []
for state in true_states:
    z_true = h_vision(state)
    noise = np.array([[np.random.randn() * vision_noise_std],
                      [np.random.randn() * vision_noise_std]])
    vision_measurements.append(z_true + noise)

print(f'Generated {len(encoder_measurements)} encoder measurements')
print(f'Generated {len(vision_measurements)} vision measurements')

In [ ]:
# ============== ENCODER-ONLY EKF ==============

x_est = x0.copy()
P_est = P0.copy()
estimates_encoder = []

for k in range(len(true_states)):
    # Predict
    x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt)
    # Update with encoder only
    x_est, P_est = ekf_update(x_pred, P_pred, encoder_measurements[k],
                               h_encoder, H_encoder, R_encoder)
    estimates_encoder.append(x_est.copy())

print(f'Encoder-only EKF complete: {len(estimates_encoder)} estimates')

In [ ]:
# ============== VISION-ONLY EKF ==============

x_est = x0.copy()
P_est = P0.copy()
estimates_vision = []

for k in range(len(true_states)):
    # Predict
    x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt)
    # Update with vision only
    x_est, P_est = ekf_update(x_pred, P_pred, vision_measurements[k],
                               h_vision, H_vision, R_vision)
    estimates_vision.append(x_est.copy())

print(f'Vision-only EKF complete: {len(estimates_vision)} estimates')

In [ ]:
# ============== COMPARE SINGLE-SENSOR BASELINES ==============

fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

for i in range(4):
    true_vals = [s[i, 0] for s in true_states]
    enc_vals = [s[i, 0] for s in estimates_encoder]
    vis_vals = [s[i, 0] for s in estimates_vision]

    axes[i].plot(times, true_vals, 'b-', lw=2, label='True')
    axes[i].plot(times, enc_vals, 'g--', lw=1.5, alpha=0.8, label='Encoder-only EKF')
    axes[i].plot(times, vis_vals, 'm:', lw=1.5, alpha=0.8, label='Vision-only EKF')
    axes[i].set_ylabel(labels[i])
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Single-Sensor EKF Baselines: Encoder-only vs Vision-only',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compute RMSE for single-sensor baselines
n_skip = int(0.5 / dt)  # Skip first 0.5s for convergence

print('Single-Sensor EKF -- RMSE (after 0.5s convergence)')
print('=' * 70)
print(f'{"State":<25} | {"Encoder-only":>12} | {"Vision-only":>12}')
print('-' * 70)

rmse_encoder = []
rmse_vision = []

for i, label in enumerate(labels):
    errors_enc = [estimates_encoder[k][i, 0] - true_states[k][i, 0]
                  for k in range(n_skip, len(estimates_encoder))]
    errors_vis = [estimates_vision[k][i, 0] - true_states[k][i, 0]
                  for k in range(n_skip, len(estimates_vision))]
    r_enc = np.sqrt(np.mean(np.array(errors_enc)**2))
    r_vis = np.sqrt(np.mean(np.array(errors_vis)**2))
    rmse_encoder.append(r_enc)
    rmse_vision.append(r_vis)
    print(f'{label:<25} | {r_enc:>12.4f} | {r_vis:>12.4f}')

print('-' * 70)
print()
print('Observations:')
print('  - Encoder-only: Precise angle tracking (direct measurement)')
print('  - Vision-only: Noisier but still converges (indirect via forward kinematics)')
print('  - Both estimate velocities via dynamics, even though neither directly measures them')

## Section 9: Multi-Rate Sensor Fusion

In practice, sensors run at different rates. Encoders might update at 100Hz (every 0.01s), while vision processes at 10Hz (every 0.1s). Multi-rate fusion uses each measurement when it arrives.

**The pattern is simple:**
1. Predict at the fastest sensor rate (encoder rate)
2. Always update with encoder (available every step)
3. Update with vision only when available (every 10th step)

This requires no special algorithm -- just conditional updates.

In [ ]:
# Multi-rate sensor configuration
dt_encoder = 0.01    # 100 Hz encoder rate
vision_interval = 10  # Vision every 10 encoder steps (10 Hz)

# Subsample true_states to encoder rate
# MuJoCo runs at dt=0.002, encoder at dt=0.01, so subsample every 5 MuJoCo steps
mujoco_steps_per_encoder = int(dt_encoder / dt)
encoder_indices = list(range(0, len(true_states), mujoco_steps_per_encoder))

true_states_encoder_rate = [true_states[i] for i in encoder_indices]
times_encoder_rate = times[encoder_indices]

print(f'Multi-rate fusion configuration:')
print(f'  Encoder rate: {1/dt_encoder:.0f} Hz (dt = {dt_encoder}s)')
print(f'  Vision rate:  {1/(dt_encoder*vision_interval):.0f} Hz (every {vision_interval} encoder steps)')
print(f'  MuJoCo dt: {dt}s, subsampling every {mujoco_steps_per_encoder} steps')
print(f'  {len(true_states_encoder_rate)} encoder-rate timesteps from {len(true_states)} MuJoCo steps')

In [ ]:
# Generate measurements at appropriate rates
np.random.seed(42)  # Same seed for fair comparison

# Encoder measurements at encoder rate (100 Hz)
encoder_meas_multirate = []
for state in true_states_encoder_rate:
    z_true = h_encoder(state)
    noise = np.array([[np.random.randn() * encoder_noise_std],
                      [np.random.randn() * encoder_noise_std]])
    encoder_meas_multirate.append(z_true + noise)

# Vision measurements at vision rate (10 Hz)
vision_meas_multirate = []
for k, state in enumerate(true_states_encoder_rate):
    if k % vision_interval == 0:
        z_true = h_vision(state)
        noise = np.array([[np.random.randn() * vision_noise_std],
                          [np.random.randn() * vision_noise_std]])
        vision_meas_multirate.append(z_true + noise)

print(f'Generated {len(encoder_meas_multirate)} encoder measurements (100 Hz)')
print(f'Generated {len(vision_meas_multirate)} vision measurements (10 Hz)')

In [ ]:
# ============== MULTI-RATE SENSOR FUSION EKF ==============

x_est = x0.copy()
P_est = P0.copy()
estimates_fused = []
covariances_fused = []
vision_update_times = []

vision_idx = 0

for k in range(len(true_states_encoder_rate)):
    # Predict at encoder rate
    x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt_encoder)
    
    # Always update with encoder
    x_est, P_est = ekf_update(x_pred, P_pred, encoder_meas_multirate[k],
                               h_encoder, H_encoder, R_encoder)
    
    # Update with vision only at vision rate
    if k % vision_interval == 0:
        x_est, P_est = ekf_update(x_est, P_est, vision_meas_multirate[vision_idx],
                                   h_vision, H_vision, R_vision)
        vision_update_times.append(times_encoder_rate[k])
        vision_idx += 1
    
    estimates_fused.append(x_est.copy())
    covariances_fused.append(P_est.copy())

print(f'Multi-rate fusion EKF complete: {len(estimates_fused)} estimates')
print(f'Vision updates at {len(vision_update_times)} timesteps')

## Section 10: Fusion Comparison

Now we compare all three approaches:
1. **Encoder-only**: Direct joint angle measurements (100 Hz)
2. **Vision-only**: End-effector position via forward kinematics (10 Hz, subsampled for fair comparison)
3. **Fused**: Both sensors at their respective rates

The key question: does fusion outperform either sensor alone?

In [ ]:
# Run encoder-only and vision-only at encoder rate for fair comparison

# Encoder-only at encoder rate
x_est = x0.copy()
P_est = P0.copy()
estimates_encoder_rate = []

for k in range(len(true_states_encoder_rate)):
    x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt_encoder)
    x_est, P_est = ekf_update(x_pred, P_pred, encoder_meas_multirate[k],
                               h_encoder, H_encoder, R_encoder)
    estimates_encoder_rate.append(x_est.copy())

# Vision-only at encoder rate (predict every step, update only at vision rate)
x_est = x0.copy()
P_est = P0.copy()
estimates_vision_rate = []
vision_idx = 0

for k in range(len(true_states_encoder_rate)):
    x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt_encoder)
    # Vision update only at vision rate
    if k % vision_interval == 0:
        x_est, P_est = ekf_update(x_pred, P_pred, vision_meas_multirate[vision_idx],
                                   h_vision, H_vision, R_vision)
        vision_idx += 1
    else:
        x_est, P_est = x_pred, P_pred  # No update, just predict
    estimates_vision_rate.append(x_est.copy())

print('Baselines at encoder rate computed.')

In [ ]:
# ============== COMPARISON PLOT ==============

fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

for i in range(4):
    true_vals = [s[i, 0] for s in true_states_encoder_rate]
    enc_vals = [s[i, 0] for s in estimates_encoder_rate]
    vis_vals = [s[i, 0] for s in estimates_vision_rate]
    fused_vals = [s[i, 0] for s in estimates_fused]

    axes[i].plot(times_encoder_rate, true_vals, 'b-', lw=2, label='True')
    axes[i].plot(times_encoder_rate, enc_vals, 'g--', lw=1.5, alpha=0.7, label='Encoder-only')
    axes[i].plot(times_encoder_rate, vis_vals, 'm:', lw=1.5, alpha=0.7, label='Vision-only (10Hz)')
    axes[i].plot(times_encoder_rate, fused_vals, 'r-', lw=1.5, alpha=0.9, label='Fused')
    
    # Mark vision update times
    for vt in vision_update_times[::5]:  # Subsample for clarity
        axes[i].axvline(x=vt, color='purple', alpha=0.1, lw=1)
    
    axes[i].set_ylabel(labels[i])
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
fig.suptitle('Multi-Rate Sensor Fusion: Encoder (100Hz) + Vision (10Hz)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Purple vertical lines indicate vision update times (10 Hz).')

In [ ]:
# ============== RMSE COMPARISON TABLE ==============

n_skip_encoder = int(0.5 / dt_encoder)  # Skip first 0.5s

print('RMSE Comparison: Encoder-only vs Vision-only vs Fused')
print('=' * 80)
print(f'{"State":<25} | {"Encoder-only":>12} | {"Vision-only":>12} | {"Fused":>12} | {"Best":>8}')
print('-' * 80)

rmse_results = {'encoder': [], 'vision': [], 'fused': []}

for i, label in enumerate(labels):
    errors_enc = [estimates_encoder_rate[k][i, 0] - true_states_encoder_rate[k][i, 0]
                  for k in range(n_skip_encoder, len(estimates_encoder_rate))]
    errors_vis = [estimates_vision_rate[k][i, 0] - true_states_encoder_rate[k][i, 0]
                  for k in range(n_skip_encoder, len(estimates_vision_rate))]
    errors_fused = [estimates_fused[k][i, 0] - true_states_encoder_rate[k][i, 0]
                    for k in range(n_skip_encoder, len(estimates_fused))]
    
    r_enc = np.sqrt(np.mean(np.array(errors_enc)**2))
    r_vis = np.sqrt(np.mean(np.array(errors_vis)**2))
    r_fused = np.sqrt(np.mean(np.array(errors_fused)**2))
    
    rmse_results['encoder'].append(r_enc)
    rmse_results['vision'].append(r_vis)
    rmse_results['fused'].append(r_fused)
    
    best = 'Fused' if r_fused <= min(r_enc, r_vis) else ('Encoder' if r_enc < r_vis else 'Vision')
    print(f'{label:<25} | {r_enc:>12.4f} | {r_vis:>12.4f} | {r_fused:>12.4f} | {best:>8}')

print('-' * 80)
print()
print('Key Insights:')
print('  1. Fusion typically matches or beats single-sensor performance')
print('  2. Fast encoder updates provide high-bandwidth tracking')
print('  3. Vision updates provide absolute position reference (prevents drift)')
print('  4. Complementary strengths: encoders are precise/fast, vision measures what we care about')

## Section 11: Interactive Exploration

Use the widget below to explore how different sensor noise levels and vision update rates affect estimation quality. This helps build intuition for sensor fusion design choices.

In [ ]:
def explore_fusion(encoder_noise=0.01, vision_noise_mm=1.0, vision_hz=10):
    """
    Run multi-rate fusion with configurable parameters and display results.
    """
    # Calculate sensor parameters
    enc_noise = encoder_noise  # rad
    vis_noise = vision_noise_mm / 1000.0  # Convert mm to m
    R_enc = np.diag([enc_noise**2, enc_noise**2])
    R_vis = np.diag([vis_noise**2, vis_noise**2])
    
    # Calculate vision interval (encoder runs at 100 Hz)
    vis_interval = max(1, int(100 / vision_hz))  # Ensure at least 1
    
    # Generate measurements with specified noise levels
    np.random.seed(42)
    enc_meas = []
    vis_meas = []
    for k, state in enumerate(true_states_encoder_rate):
        z_enc = h_encoder(state) + np.random.randn(2, 1) * enc_noise
        enc_meas.append(z_enc)
        if k % vis_interval == 0:
            z_vis = h_vision(state) + np.random.randn(2, 1) * vis_noise
            vis_meas.append(z_vis)
    
    # Run fusion EKF
    x_est = x0.copy()
    P_est = P0.copy()
    estimates = []
    vis_idx = 0
    
    for k in range(len(true_states_encoder_rate)):
        x_pred, P_pred = ekf_predict(x_est, P_est, arm_f, F_func, Q, dt_encoder)
        x_est, P_est = ekf_update(x_pred, P_pred, enc_meas[k], h_encoder, H_encoder, R_enc)
        if k % vis_interval == 0:
            x_est, P_est = ekf_update(x_est, P_est, vis_meas[vis_idx], h_vision, H_vision, R_vis)
            vis_idx += 1
        estimates.append(x_est.copy())
    
    # Compute RMSE
    n_skip = int(0.5 / dt_encoder)
    rmse = []
    for i in range(4):
        errors = [estimates[k][i, 0] - true_states_encoder_rate[k][i, 0]
                  for k in range(n_skip, len(estimates))]
        rmse.append(np.sqrt(np.mean(np.array(errors)**2)))
    
    # Plot results
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    for i, ax in enumerate(axes.flat):
        true_vals = [s[i, 0] for s in true_states_encoder_rate]
        est_vals = [s[i, 0] for s in estimates]
        
        ax.plot(times_encoder_rate, true_vals, 'b-', lw=2, label='True')
        ax.plot(times_encoder_rate, est_vals, 'r-', lw=1.5, alpha=0.9, label='Fused')
        ax.set_ylabel(labels[i])
        ax.set_title(f'{labels[i]} (RMSE: {rmse[i]:.4f})')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
    
    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 1].set_xlabel('Time (s)')
    
    fig.suptitle(f'Sensor Fusion: Encoder noise={enc_noise:.3f} rad, '
                 f'Vision noise={vision_noise_mm:.1f} mm, Vision rate={vision_hz} Hz',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f'\nConfiguration:')
    print(f'  Encoder noise: {enc_noise:.4f} rad ({np.degrees(enc_noise):.2f} deg)')
    print(f'  Vision noise:  {vision_noise_mm:.1f} mm')
    print(f'  Vision rate:   {vision_hz} Hz (every {vis_interval} encoder steps)')
    print(f'\nRMSE Summary:')
    for i, label in enumerate(labels):
        print(f'  {label}: {rmse[i]:.4f}')


# Interactive widget
interact_manual(
    explore_fusion,
    encoder_noise=FloatLogSlider(value=0.01, base=10, min=-3, max=0, step=0.1,
                                  description='Encoder (rad)'),
    vision_noise_mm=FloatLogSlider(value=1.0, base=10, min=-1, max=2, step=0.1,
                                    description='Vision (mm)'),
    vision_hz=IntSlider(value=10, min=1, max=100, step=1,
                         description='Vision Hz')
)

## Summary

This notebook demonstrated **multi-rate sensor fusion** on a 2-DOF planar robot arm:

### Key Concepts

1. **Different sensors measure different things**: Encoders measure joint angles directly, vision measures end-effector position in Cartesian coordinates.

2. **Forward kinematics bridges joint and Cartesian space**: The vision measurement model uses forward kinematics to relate joint angles to end-effector position.

3. **Multi-rate fusion is simple**: No special algorithm required -- just predict at the fastest rate and update with each sensor when available.

4. **Complementary sensor strengths**: Fast/precise encoders + slower/absolute vision = better overall performance.

### Next Steps

- **Notebook 05**: Add IMU sensors and explore how acceleration measurements help track velocity (another complementary sensor type).

- **Advanced topics**: Out-of-sequence measurements, sensor dropout handling, delayed measurements.